## Esperanza de Bellman (Bellman Expectation Equation)

La Ecuación de Esperanza de Bellman es fundamental en el aprendizaje por refuerzo. Describe la relación entre el valor de un estado (o un par estado-acción) y el valor de los estados subsiguientes, bajo una política dada. Básicamente, afirma que el valor de un estado (o estado-acción) es la recompensa esperada por tomar una acción desde ese estado, más el valor descontado del siguiente estado. Es una base para algoritmos que buscan encontrar la función de valor óptima.

### Ecuación del Valor de un Estado (V-value):

$V^{\pi}(s) = \sum_{a \in A} \pi(a|s) \sum_{s' \in S, r \in R} P(s', r | s, a) [r + \gamma V^{\pi}(s')]$

## Métodos Tabulares

Los métodos tabulares en el aprendizaje por refuerzo se refieren a algoritmos que almacenan las funciones de valor (V-value o Q-value) en tablas. Esto es factible cuando el espacio de estados y acciones es lo suficientemente pequeño como para ser enumerado y almacenado en memoria. Estos métodos son la base de muchos algoritmos de RL y son muy efectivos en entornos simples.

**Ejemplos:**
*   **Evaluación de Políticas:** Monte Carlo, Temporal Difference (TD) (como TD(0), SARSA, Q-Learning)
*   **Control de Políticas:** Q-Learning, SARSA

## Q-Learning

Q-Learning es un algoritmo de aprendizaje por refuerzo *off-policy* que busca aprender una función de valor de acción óptima, $Q^*(s,a)$. "Off-policy" significa que aprende la política óptima independientemente de la política que está siguiendo el agente para explorar el entorno. Utiliza la siguiente regla de actualización:

$Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$

Donde:
*   $Q(s,a)$ es el valor Q para el estado $s$ y la acción $a$.
*   $\alpha$ es la tasa de aprendizaje.
*   $r$ es la recompensa recibida.
*   $\gamma$ es el factor de descuento.
*   $s'$ es el siguiente estado.
*   $\max_{a'} Q(s',a')$ es el máximo valor Q para el siguiente estado $s'$, que representa la acción óptima que se tomaría desde $s'$.

### Ejemplo Simple de Q-Learning (Pseudocódigo/Concepto)

In [1]:
import numpy as np

# Entorno de ejemplo: Rejilla de 2x2
# 0: Camino, -1: Obstáculo, 1: Meta
# Recompensas: -0.1 por paso, +10 en la meta
environment = np.array([
    [0, 0],
    [-1, 1]
])

# Acciones: 0: Arriba, 1: Abajo, 2: Izquierda, 3: Derecha
# Convenciones para el estado: (fila, columna)

# Inicializar la tabla Q (estados x acciones)
# Estados: (0,0), (0,1), (1,0), (1,1) -> 4 estados
# Acciones: 4 acciones posibles
num_states = 4 # Mapeamos (0,0) -> 0, (0,1) -> 1, (1,0) -> 2, (1,1) -> 3
num_actions = 4
q_table = np.zeros((num_states, num_actions))

# Parámetros de Q-Learning
learning_rate = 0.8
discount_factor = 0.95
episodes = 1000
epsilon = 0.1 # Para exploración (epsilon-greedy)

def state_to_index(row, col):
    return row * environment.shape[1] + col

def index_to_state(index):
    return index // environment.shape[1], index % environment.shape[1]

# Función para tomar una acción (epsilon-greedy)
def choose_action(state_idx):
    if np.random.uniform(0, 1) < epsilon:
        return np.random.randint(num_actions) # Exploración
    else:
        return np.argmax(q_table[state_idx, :]) # Explotación

# Simulación de episodios
for episode in range(episodes):
    current_row, current_col = 0, 0 # Iniciar en (0,0)
    current_state_idx = state_to_index(current_row, current_col)

    while True:
        action = choose_action(current_state_idx)

        # Simular el siguiente estado y recompensa
        next_row, next_col = current_row, current_col
        reward = -0.1 # Recompensa por paso
        done = False

        if action == 0: # Arriba
            next_row = max(0, current_row - 1)
        elif action == 1: # Abajo
            next_row = min(environment.shape[0] - 1, current_row + 1)
        elif action == 2: # Izquierda
            next_col = max(0, current_col - 1)
        elif action == 3: # Derecha
            next_col = min(environment.shape[1] - 1, current_col + 1)

        # Recompensas específicas y terminales
        if environment[next_row, next_col] == -1: # Obstáculo
            reward = -10 # Penalización por chocar con el obstáculo
            next_row, next_col = current_row, current_col # Permanece en el mismo lugar
        elif environment[next_row, next_col] == 1: # Meta
            reward = 10
            done = True

        next_state_idx = state_to_index(next_row, next_col)

        # Actualización de Q-Learning
        old_q_value = q_table[current_state_idx, action]
        next_max_q = np.max(q_table[next_state_idx, :]) # Elige el máximo Q para el siguiente estado

        new_q_value = old_q_value + learning_rate * (reward + discount_factor * next_max_q - old_q_value)
        q_table[current_state_idx, action] = new_q_value

        current_row, current_col = next_row, next_col
        current_state_idx = next_state_idx

        if done: # o si se alcanza un número máximo de pasos
            break

print("Tabla Q Final:")
print(q_table)

print("\nPolítica óptima (aproximada):")
for i in range(num_states):
    r, c = index_to_state(i)
    if environment[r, c] == -1: # No hay política para obstáculos
        print(f"Estado ({r},{c}): Obstáculo")
        continue
    best_action_idx = np.argmax(q_table[i, :])
    actions_map = {0: "Arriba", 1: "Abajo", 2: "Izquierda", 3: "Derecha"}
    print(f"Estado ({r},{c}): {actions_map[best_action_idx]}")


Tabla Q Final:
[[ 8.83 -1.07  8.83  9.4 ]
 [ 9.4  10.    8.83  9.4 ]
 [ 0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.  ]]

Política óptima (aproximada):
Estado (0,0): Derecha
Estado (0,1): Abajo
Estado (1,0): Obstáculo
Estado (1,1): Arriba


## SARSA (State-Action-Reward-State-Action)

SARSA es un algoritmo de aprendizaje por refuerzo *on-policy*. A diferencia de Q-Learning, que es *off-policy*, SARSA aprende la función de valor de acción para la política que está siendo seguida por el agente. Esto significa que la estimación del valor $Q(s',a')$ se basa en la *siguiente acción $a'$ realmente tomada* por la política actual, no en la acción óptima.

La regla de actualización de SARSA es:

$Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma Q(s',a') - Q(s,a)]$

Donde la principal diferencia con Q-Learning es que $Q(s',a')$ se utiliza en lugar de $\max_{a'} Q(s',a')$.

**Diferencia Clave con Q-Learning:**
*   **Q-Learning (Off-policy):** Asume que la siguiente acción *óptima* se tomará desde $s'$ (usa `max(Q(s', a'))`). Es más optimista y puede aprender más rápido, pero puede ser más arriesgado en entornos con peligros terminales si la política de exploración es demasiado exploratoria.
*   **SARSA (On-policy):** Considera la *acción real* $a'$ que el agente tomará en $s'$ (usa `Q(s', a')`). Esto hace que SARSA sea más conservador y aprenda una política más segura, ya que tiene en cuenta las acciones subóptimas que podría tomar durante la exploración.

In [2]:
import numpy as np

# Entorno de ejemplo: Rejilla de 2x2 (el mismo que Q-Learning)
# 0: Camino, -1: Obstáculo, 1: Meta
# Recompensas: -0.1 por paso, +10 en la meta
environment = np.array([
    [0, 0],
    [-1, 1]
])

# Acciones: 0: Arriba, 1: Abajo, 2: Izquierda, 3: Derecha
num_states = 4 # Mapeamos (0,0) -> 0, (0,1) -> 1, (1,0) -> 2, (1,1) -> 3
num_actions = 4
q_table = np.zeros((num_states, num_actions))

# Parámetros de SARSA
learning_rate = 0.8
discount_factor = 0.95
episodes = 1000
epsilon = 0.1 # Para exploración (epsilon-greedy)

def state_to_index(row, col):
    return row * environment.shape[1] + col

def index_to_state(index):
    return index // environment.shape[1], index % environment.shape[1]

# Función para tomar una acción (epsilon-greedy)
def choose_action_sarsa(state_idx):
    if np.random.uniform(0, 1) < epsilon:
        return np.random.randint(num_actions) # Exploración
    else:
        return np.argmax(q_table[state_idx, :]) # Explotación (basada en la Q actual)

# Simulación de episodios para SARSA
for episode in range(episodes):
    current_row, current_col = 0, 0 # Iniciar en (0,0)
    current_state_idx = state_to_index(current_row, current_col)
    action = choose_action_sarsa(current_state_idx) # Primera acción elegida por la política actual

    while True:
        # Simular el siguiente estado y recompensa
        next_row, next_col = current_row, current_col
        reward = -0.1 # Recompensa por paso
        done = False

        if action == 0: # Arriba
            next_row = max(0, current_row - 1)
        elif action == 1: # Abajo
            next_row = min(environment.shape[0] - 1, current_row + 1)
        elif action == 2: # Izquierda
            next_col = max(0, current_col - 1)
        elif action == 3: # Derecha
            next_col = min(environment.shape[1] - 1, current_col + 1)

        # Recompensas específicas y terminales
        if environment[next_row, next_col] == -1: # Obstáculo
            reward = -10 # Penalización por chocar con el obstáculo
            next_row, next_col = current_row, current_col # Permanece en el mismo lugar
        elif environment[next_row, next_col] == 1: # Meta
            reward = 10
            done = True

        next_state_idx = state_to_index(next_row, next_col)

        # La principal diferencia: elige la siguiente acción a' según la política actual
        next_action = choose_action_sarsa(next_state_idx)

        # Actualización de SARSA
        old_q_value = q_table[current_state_idx, action]

        # Aquí usamos Q(s',a') donde a' es la acción *realmente* tomada (o elegida por la política de exploración)
        next_q_value = q_table[next_state_idx, next_action]

        new_q_value = old_q_value + learning_rate * (reward + discount_factor * next_q_value - old_q_value)
        q_table[current_state_idx, action] = new_q_value

        current_row, current_col = next_row, next_col
        current_state_idx = next_state_idx
        action = next_action # La acción para el siguiente paso ya ha sido elegida

        if done:
            break

print("Tabla Q Final (SARSA):")
print(q_table)

print("\nPolítica óptima (aproximada):")
for i in range(num_states):
    r, c = index_to_state(i)
    if environment[r, c] == -1: # No hay política para obstáculos
        print(f"Estado ({r},{c}): Obstáculo")
        continue
    best_action_idx = np.argmax(q_table[i, :])
    actions_map = {0: "Arriba", 1: "Abajo", 2: "Izquierda", 3: "Derecha"}
    print(f"Estado ({r},{c}): {actions_map[best_action_idx]}")


Tabla Q Final (SARSA):
[[ 7.73004542 -1.28320814  8.50703435  9.4       ]
 [ 9.4        10.          8.15051033  8.05360151]
 [ 0.          0.          0.          0.        ]
 [ 0.          0.          0.          0.        ]]

Política óptima (aproximada):
Estado (0,0): Derecha
Estado (0,1): Abajo
Estado (1,0): Obstáculo
Estado (1,1): Arriba


## Deep Q-Networks (DQN)

DQN (Deep Q-Network) extiende Q-Learning a entornos con espacios de estados muy grandes o continuos, donde las tablas Q no son viables. En lugar de una tabla, DQN utiliza una red neuronal profunda (DQN) para aproximar la función Q. Esto permite que el agente aprenda a partir de experiencias y generalice a estados no vistos previamente.

**Componentes Clave de DQN:**
1.  **Red Neuronal Profunda:** Sustituye la tabla Q por una CNN (o feed-forward) que toma el estado como entrada y produce valores Q para cada acción como salida.
2.  **Experiencia de Replay (Experience Replay):** Almacena las transiciones `(s, a, r, s')` en un *replay buffer*. Durante el entrenamiento, se muestrean aleatoriamente lotes de estas experiencias para actualizar la red. Esto rompe la correlación entre muestras consecutivas y mejora la estabilidad del aprendizaje.
3.  **Red Objetivo (Target Network):** Para estabilizar el aprendizaje, DQN utiliza dos redes: una red de Q principal (que se entrena continuamente) y una red objetivo (una copia de la red principal que se actualiza periódicamente). La red objetivo se usa para calcular el valor objetivo ($r + \gamma \max_{a'} Q_{target}(s',a')$), lo que reduce la inestabilidad de la actualización de Q.

### Regla de Actualización (con red objetivo):

La pérdida se calcula como:

$L = (Q(s,a;\theta) - y)^2$

Donde $y = r + \gamma \max_{a'} Q_{target}(s',a')$ es el objetivo de Q, y $\theta$ son los pesos de la red principal.

### Estructura Conceptual del Código DQN

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
import random
from collections import deque

# --- 1. Definir la Red Neuronal (Q-Network) ---
def create_q_model(state_shape, action_space):
    model = models.Sequential()
    model.add(layers.Input(shape=state_shape))
    model.add(layers.Dense(24, activation='relu'))
    model.add(layers.Dense(24, activation='relu'))
    model.add(layers.Dense(action_space, activation='linear')) # Salida: Q-values para cada acción
    return model

# --- 2. Experiencia de Replay Buffer ---
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

# --- 3. Clase Agente DQN (conceptual) ---
class DQNAgent:
    def __init__(self, state_shape, action_space):
        self.state_shape = state_shape
        self.action_space = action_space
        self.gamma = 0.99  # Factor de descuento
        self.epsilon = 1.0  # Epsilon para exploración
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.batch_size = 32
        self.memory = ReplayBuffer(capacity=2000)

        self.model = create_q_model(state_shape, action_space) # Red principal
        self.target_model = create_q_model(state_shape, action_space) # Red objetivo
        self.target_model.set_weights(self.model.get_weights())

        self.optimizer = tf.keras.optimizers.Adam(learning_rate=self.learning_rate)
        self.loss_fn = tf.keras.losses.MeanSquaredError()

    def choose_action(self, state):
        if np.random.rand() <= self.epsilon:
            return np.random.randint(self.action_space) # Exploración
        q_values = self.model.predict(state[np.newaxis, :], verbose=0) # Explotación
        return np.argmax(q_values[0])

    def remember(self, state, action, reward, next_state, done):
        self.memory.add(state, action, reward, next_state, done)

    def replay(self):
        if len(self.memory) < self.batch_size:
            return

        minibatch = self.memory.sample(self.batch_size)
        states, actions, rewards, next_states, dones = zip(*minibatch)

        states = np.array(states)
        next_states = np.array(next_states)

        # Calcular los Q-values objetivo
        target_q_values = self.target_model.predict(next_states, verbose=0)
        max_next_q_values = np.max(target_q_values, axis=1)

        targets = rewards + self.gamma * max_next_q_values * (1 - np.array(dones))

        # Obtener los Q-values actuales de la red principal
        current_q_values = self.model.predict(states, verbose=0)

        # Clonar para modificar solo las acciones tomadas
        target_f = np.copy(current_q_values)
        for i, action in enumerate(actions):
            target_f[i][action] = targets[i] # Solo actualizamos la acción que se tomó

        # Entrenar la red principal
        self.model.train_on_batch(states, target_f)

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())

# --- Uso conceptual (no se ejecuta completamente sin un entorno real) ---
# Si tuviéramos un entorno como Gym:
# import gym
# env = gym.make('CartPole-v1')
# state_shape = env.observation_space.shape
# action_space = env.action_space.n

# Para este ejemplo, simulamos:
state_shape_example = (4,) # Por ejemplo, 4 observaciones de un estado
action_space_example = 2 # Por ejemplo, 2 acciones posibles

agent = DQNAgent(state_shape_example, action_space_example)

print("Modelo principal:")
agent.model.summary()
print("\nModelo objetivo:")
agent.target_model.summary()

print("\nEjemplo de adición al buffer de experiencia (simulado):")
example_state = np.random.rand(*state_shape_example)
example_next_state = np.random.rand(*state_shape_example)
agent.remember(example_state, 0, 1, example_next_state, False)
print(f"Tamaño del buffer de experiencia: {len(agent.memory)}")


Modelo principal:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 24)             │           120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │           600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            50 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 770 (3.01 KB)

 Trainable params: 770 (3.01 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 24)             │           120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 24)             │           600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2)              │            50 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 770 (3.01 KB)

 Trainable params: 770 (3.01 KB)

 Non-trainable params: 0 (0.00 B)


Ejemplo de adición al buffer de experiencia (simulado):
Tamaño del buffer de experiencia: 1


## Proximal Policy Optimization (PPO)

PPO es un algoritmo de *Policy Gradient* (Gradiente de Políticas) que se ha vuelto muy popular debido a su buen rendimiento y relativa simplicidad de implementación. A diferencia de los métodos basados en valor (como Q-Learning y DQN) que aprenden una función de valor, los métodos de gradiente de políticas aprenden directamente una función de política que mapea estados a acciones.

**Problema con Policy Gradients puros:** La actualización de políticas puede ser muy inestable, ya que un pequeño cambio en los pesos puede llevar a un cambio drástico en la política y, por lo tanto, en el rendimiento del agente.

**PPO Aborda la Inestabilidad:** PPO introduce un mecanismo de *clip* (recorte) en la función de objetivo para evitar actualizaciones de política demasiado grandes. Esto asegura que la nueva política no se desvíe demasiado de la política anterior, lo que mejora la estabilidad del entrenamiento.

**Componentes Clave de PPO:**
1.  **Redes de Actor-Crítico:** Típicamente, PPO utiliza una arquitectura actor-crítico:
    *   **Actor (Policy Network):** Una red neuronal que toma el estado como entrada y produce las probabilidades de acción (o los parámetros de una distribución de acciones en entornos continuos).
    *   **Crítico (Value Network):** Una red neuronal que toma el estado como entrada y predice el valor esperado del estado (función de valor $V(s)$). El crítico ayuda al actor al estimar el *advantage* ($A(s,a) = Q(s,a) - V(s)$), que indica cuán buena es una acción dada en un estado en comparación con el promedio.
2.  **Función de Pérdida con Clip:** La función de pérdida de PPO se diseña para penalizar los cambios excesivos en la política. Utiliza una relación de probabilidad (ratio) entre la nueva política y la política antigua, y recorta esta relación para mantener las actualizaciones dentro de un rango razonable.

### Función de Pérdida con Clip (Conceptual):

$L^{CLIP}(\theta) = \mathbb{E}_t[\min(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t)]$

Donde:
*   $r_t(\theta) = \frac{\pi_{\theta}(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ es la relación de probabilidad.
*   $A_t$ es la estimación de la ventaja en el tiempo $t$.
*   $\epsilon$ es un hiperparámetro de clipping (típicamente 0.1 o 0.2).

### Estructura Conceptual del Código PPO

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# --- 1. Definir la Red del Actor (Policy Network) ---
def create_actor_model(state_shape, action_space):
    model = models.Sequential()
    model.add(layers.Input(shape=state_shape))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(64, activation='relu'))
    # Salida: probabilidades de cada acción (softmax para discreto)
    model.add(layers.Dense(action_space, activation='softmax'))
    return model

# --- 2. Definir la Red del Crítico (Value Network) ---
def create_critic_model(state_shape):
    model = models.Sequential()
    model.add(layers.Input(shape=state_shape))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(1, activation='linear')) # Salida: valor del estado
    return model

# --- Clase Agente PPO (conceptual) ---
class PPOAgent:
    def __init__(self, state_shape, action_space):
        self.state_shape = state_shape
        self.action_space = action_space
        self.gamma = 0.99 # Factor de descuento
        self.lambda_gae = 0.95 # Factor para Generalized Advantage Estimation
        self.clip_ratio = 0.2 # Hiperparámetro de clipping
        self.actor_lr = 0.0003 # Tasa de aprendizaje del actor
        self.critic_lr = 0.001 # Tasa de aprendizaje del crítico

        self.actor = create_actor_model(state_shape, action_space)
        self.critic = create_critic_model(state_shape)

        self.actor_optimizer = tf.keras.optimizers.Adam(learning_rate=self.actor_lr)
        self.critic_optimizer = tf.keras.optimizers.Adam(learning_rate=self.critic_lr)

    def choose_action(self, state):
        # Añade una dimensión para el batch (si el estado no es un batch)
        state = np.asarray(state).astype(np.float32)
        state = np.expand_dims(state, axis=0)

        prob_dist = self.actor.predict(state, verbose=0)[0] # Obtener probabilidades de acción
        action = np.random.choice(self.action_space, p=prob_dist) # Muestrear una acción
        return action, prob_dist[action]

    def compute_advantages(self, rewards, values, next_values, dones):
        # Cálculo de las Ventajas Generalizadas (GAE)
        advantages = np.zeros_like(rewards, dtype=np.float32)
        last_gae = 0
        for t in reversed(range(len(rewards))):
            delta = rewards[t] + self.gamma * next_values[t] * (1 - dones[t]) - values[t]
            advantages[t] = delta + self.gamma * self.lambda_gae * (1 - dones[t]) * last_gae
            last_gae = advantages[t]
        return advantages

    def train(self, states, actions, old_probs, advantages, returns):
        # --- Entrenamiento del Actor ---
        with tf.GradientTape() as tape:
            current_probs = self.actor(states)
            # Obtener las probabilidades de las acciones tomadas
            current_action_probs = tf.gather_nd(current_probs,
                                                 tf.stack([tf.range(tf.shape(actions)[0]), actions], axis=1))

            ratio = current_action_probs / (old_probs + 1e-10) # Evitar división por cero

            # Implementar la función de pérdida PPO con clipping
            pg_loss1 = ratio * advantages
            pg_loss2 = tf.clip_by_value(ratio, 1 - self.clip_ratio, 1 + self.clip_ratio) * advantages
            actor_loss = -tf.reduce_mean(tf.minimum(pg_loss1, pg_loss2))

        actor_grads = tape.gradient(actor_loss, self.actor.trainable_variables)
        self.actor_optimizer.apply_gradients(zip(actor_grads, self.actor.trainable_variables))

        # --- Entrenamiento del Crítico ---
        with tf.GradientTape() as tape:
            predicted_values = self.critic(states)
            critic_loss = tf.reduce_mean(tf.square(returns - predicted_values)) # MSE

        critic_grads = tape.gradient(critic_loss, self.critic.trainable_variables)
        self.critic_optimizer.apply_gradients(zip(critic_grads, self.critic.trainable_variables))

# --- Uso conceptual (no se ejecuta completamente sin un entorno real) ---
# Si tuviéramos un entorno como Gym:
# import gym
# env = gym.make('CartPole-v1')
# state_shape = env.observation_space.shape
# action_space = env.action_space.n

# Para este ejemplo, simulamos:
state_shape_example = (4,)
action_space_example = 2

agent_ppo = PPOAgent(state_shape_example, action_space_example)

print("Modelo Actor:")
agent_ppo.actor.summary()
print("\nModelo Crítico:")
agent_ppo.critic.summary()

print("\nEjemplo de una acción elegida (simulada):")
example_state_ppo = np.random.rand(*state_shape_example)
action_ppo, prob_ppo = agent_ppo.choose_action(example_state_ppo)
print(f"Estado: {example_state_ppo}, Acción elegida: {action_ppo}, Probabilidad: {prob_ppo:.4f}")

# Ejemplo de datos para entrenamiento (simulados)
sim_states = np.random.rand(32, *state_shape_example).astype(np.float32)
sim_actions = np.random.randint(0, action_space_example, size=32)
sim_old_probs = np.random.rand(32) # Probabilidades de la política antigua
sim_advantages = np.random.rand(32) - 0.5 # Ventajas (pueden ser positivas o negativas)
sim_returns = np.random.rand(32)

print("\nIntentando un paso de entrenamiento (simulado):")
# agent_ppo.train(sim_states, sim_actions, sim_old_probs, sim_advantages, sim_returns)
# Para ejecutar esto, se necesitarían datos reales de un entorno y un ciclo de entrenamiento completo.
print("El entrenamiento PPO requiere un ciclo de recolección de experiencias y actualización.")


Modelo Actor:


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,610 (18.01 KB)

 Trainable params: 4,610 (18.01 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,545 (17.75 KB)

 Trainable params: 4,545 (17.75 KB)

 Non-trainable params: 0 (0.00 B)


Ejemplo de una acción elegida (simulada):
Estado: [0.59087686 0.04047413 0.47990536 0.67561302], Acción elegida: 0, Probabilidad: 0.4758

Intentando un paso de entrenamiento (simulado):
El entrenamiento PPO requiere un ciclo de recolección de experiencias y actualización.
